# 实践项目 00：MNIST 手写数字分类

我们将在 Kaggle Notebook 中读取 MNIST，完成数据检查、小型卷积神经网络训练、测试集评价和一次噪声稳健性比较。

Kaggle 是本项目的首选实践入口。打开公开 Notebook 后，先点击“复制并编辑”保存到自己的账户，再逐步运行、修改代码并观察自己的输出；下载 Notebook 到电脑运行是补充方式。
代码填写位置：标有 TODO，或明确写成 pass、None 占位的代码位置，以及标有“你的回答”的 Markdown 单元格，就是需要完成的部分。先阅读当前任务说明，再根据变量名、输入输出 shape、注释和下一步的 print/assert 填写；不要直接把参考结果数字写进代码。
完成一个任务后，先运行当前单元格和后续检查单元格，确认输出形状、指标和输出文件符合说明，再进入下一项。

## 实践任务
1. 读取图像和标签，核对 shape、像素范围与类别分布
2. 根据训练集统计量完成归一化
3. 补全小型 CNN 的卷积模块与分类层
4. 补全一个标准训练步骤并记录训练曲线
5. 在测试集上计算准确率、每类 F1 与混淆矩阵
6. 加入高斯噪声并比较模型性能
7. 只修改一个训练参数，完成一次对照实验

## 需要保存的结果
- `task0_data_visualization.png`
- `task0_training_curve.png`
- `task0_confusion_matrix.png`
- `task0_result.json`


## 实践顺序

1. 读取 MNIST 并核对张量形状、数据类型、像素范围和标签分布。
2. 计算训练图像的均值与标准差，完成归一化。
3. 补全小型卷积神经网络的第二个卷积块与分类层。
4. 补全一个标准 PyTorch 训练步骤。
5. 在独立测试集上计算准确率、每类 F1 和混淆矩阵。
6. 对测试图像加入高斯噪声，比较原始图像与扰动图像的性能。
7. 修改一个训练参数，比较验证结果并写出结论。

建议先完整阅读当前任务，再按顺序运行。


In [ ]:
# 0. 导入库与固定随机性
from pathlib import Path
import json, random, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision import datasets, transforms

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
WORKDIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
print('device =', device)
print('working directory =', WORKDIR)

## 1. 读取数据

优先读取 Kaggle 已挂载的手写数字 CSV；若当前环境没有该数据，则尝试使用 `torchvision.datasets.MNIST`。最后的本地回退数据只用于环境检查，本项目使用 MNIST，并在 `task0_result.json` 中确认 `dataset_name`。


In [ ]:
def load_digit_data():
    """返回 train_dataset, test_dataset, dataset_name。图像范围统一为 [0, 1]，形状为 [1, 28, 28]。"""
    # Kaggle Digit Recognizer 常见路径
    csv_candidates = list(Path('/kaggle/input').glob('**/train.csv')) if Path('/kaggle/input').exists() else []
    for csv_path in csv_candidates:
        try:
            df = pd.read_csv(csv_path)
            pixel_cols = [c for c in df.columns if str(c).startswith('pixel')]
            if 'label' in df.columns and len(pixel_cols) == 784:
                x = torch.tensor(df[pixel_cols].to_numpy(), dtype=torch.float32).reshape(-1, 1, 28, 28) / 255.0
                y = torch.tensor(df['label'].to_numpy(), dtype=torch.long)
                g = torch.Generator().manual_seed(SEED)
                n_test = max(1000, int(0.15 * len(x)))
                perm = torch.randperm(len(x), generator=g)
                test_idx, train_idx = perm[:n_test], perm[n_test:]
                return TensorDataset(x[train_idx], y[train_idx]), TensorDataset(x[test_idx], y[test_idx]), 'MNIST / Kaggle Digit Recognizer'
        except Exception:
            pass

    # 标准 torchvision MNIST
    try:
        tfm = transforms.ToTensor()
        root = WORKDIR / 'mnist_data'
        train_ds = datasets.MNIST(root=root, train=True, transform=tfm, download=True)
        test_ds = datasets.MNIST(root=root, train=False, transform=tfm, download=True)
        return train_ds, test_ds, 'MNIST / torchvision'
    except Exception as exc:
        warnings.warn(f'MNIST 暂时无法读取，使用 sklearn digits 完成本地结构检查：{exc}')

    # 本地回退，仅用于检查 Notebook 结构
    from sklearn.datasets import load_digits
    from sklearn.model_selection import train_test_split
    import torch.nn.functional as F
    d = load_digits()
    x = torch.tensor(d.images, dtype=torch.float32).unsqueeze(1) / 16.0
    x = F.interpolate(x, size=(28, 28), mode='bilinear', align_corners=False)
    y = torch.tensor(d.target, dtype=torch.long)
    idx = np.arange(len(y))
    tr, te = train_test_split(idx, test_size=0.2, random_state=SEED, stratify=y.numpy())
    return TensorDataset(x[tr], y[tr]), TensorDataset(x[te], y[te]), 'sklearn digits fallback（仅环境检查）'

train_full, test_dataset, dataset_name = load_digit_data()
print(dataset_name)
print('train_full =', len(train_full), 'test =', len(test_dataset))

## 任务 1：数据检查与真实样本可视化

完成下一个代码块中的统计量。需要说明 `[1, 28, 28]` 三个数字分别表示什么，并检查标签是否覆盖 0–9。随后保存真实样本与类别分布图。


In [ ]:
# TODO 1：完成数据检查
sample_image, sample_label = train_full[0]

sample_count = None            # TODO：训练数据样本数
image_shape = None             # TODO：单张图像 shape
image_dtype = None             # TODO：数据类型
pixel_min = None               # TODO：像素最小值，转成 Python float
pixel_max = None               # TODO：像素最大值，转成 Python float

all_labels = torch.tensor([int(train_full[i][1]) for i in range(len(train_full))])
class_counts = None            # TODO：长度为 10 的类别计数张量

print('sample_count =', sample_count)
print('image_shape =', image_shape)
print('image_dtype =', image_dtype)
print('pixel range =', pixel_min, pixel_max)
print('label range =', int(all_labels.min()), int(all_labels.max()))
print('class_counts =', class_counts)

assert sample_count == len(train_full)
assert tuple(image_shape) == (1, 28, 28)
assert pixel_min >= 0 and pixel_max <= 1
assert len(class_counts) == 10

In [ ]:
# 真实样本与类别分布
fig, axes = plt.subplots(3, 7, figsize=(12, 6))
for ax, idx in zip(axes.ravel()[:20], np.linspace(0, len(train_full)-1, 20, dtype=int)):
    image, label = train_full[idx]
    ax.imshow(image.squeeze().numpy(), cmap='gray')
    ax.set_title(f'label={int(label)}')
    ax.axis('off')
for ax in axes.ravel()[20:]:
    ax.axis('off')
plt.tight_layout()
plt.savefig(WORKDIR/'task0_data_visualization.png', dpi=180, bbox_inches='tight')
plt.show()

plt.figure(figsize=(8, 4))
plt.bar(np.arange(10), class_counts.numpy())
plt.xticks(np.arange(10))
plt.xlabel('digit label')
plt.ylabel('sample count')
plt.title('Training label distribution')
plt.show()

## 2. 划分训练集和验证集

验证集用于选择模型和训练设置，测试集只在全部选择完成后评价一次。这里固定随机种子，保持不同同学之间的划分可比较。


In [ ]:
val_size = max(1000, int(0.15 * len(train_full))) if len(train_full) > 5000 else max(200, int(0.15 * len(train_full)))
train_size = len(train_full) - val_size
train_dataset, val_dataset = random_split(
    train_full, [train_size, val_size], generator=torch.Generator().manual_seed(SEED)
)
print('train =', len(train_dataset), 'validation =', len(val_dataset), 'test =', len(test_dataset))

## 任务 2：训练集归一化

归一化参数只能由训练集计算。完成均值与标准差的计算，再让三个数据集使用同一组参数。这里不修改原始数据文件，只在 DataLoader 输出后进行标准化。


In [ ]:
# TODO 2：从训练子集计算像素均值与标准差
stat_loader = DataLoader(train_dataset, batch_size=256, shuffle=False)
pixel_sum = 0.0
pixel_sq_sum = 0.0
pixel_count = 0

for images, _ in stat_loader:
    # TODO：累计 images 的总和、平方和与像素数量
    pass

train_mean = None  # TODO
train_std = None   # TODO，使用 E[x²] - E[x]²
print('train_mean =', train_mean, 'train_std =', train_std)
assert 0 < train_mean < 1
assert train_std > 0

def normalize_batch(images):
    return (images - train_mean) / (train_std + 1e-8)

BATCH_SIZE = 128
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

## 任务 3：补全小型卷积神经网络

输入张量为 `[batch, 1, 28, 28]`。第一个卷积块把通道数变为 16，并通过池化把空间尺寸变为 14×14。需要补全第二个卷积块和分类层，使最终输出形状为 `[batch, 10]`。


In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            # TODO 3A：加入 16→32 的 3×3 卷积、ReLU 和 2×2 最大池化
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            # TODO 3B：加入 32×7×7 → 64 的全连接层、ReLU，以及 64 → 10 的输出层
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model = SmallCNN().to(device)
example = torch.zeros(4, 1, 28, 28, device=device)
with torch.no_grad():
    output = model(example)
print(model)
print('output shape =', tuple(output.shape))
print('parameter count =', sum(p.numel() for p in model.parameters()))
assert tuple(output.shape) == (4, 10)

## 任务 4：补全训练步骤

一个标准训练批次依次执行：清空旧梯度、前向传播、计算损失、反向传播、更新参数。每一轮训练后在验证集上计算准确率。


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def evaluate(model, loader, noise_sigma=0.0):
    model.eval()
    ys, ps = [], []
    total_loss = 0.0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            if noise_sigma > 0:
                images = torch.clamp(images + noise_sigma * torch.randn_like(images), 0, 1)
            logits = model(normalize_batch(images))
            total_loss += criterion(logits, labels).item() * len(labels)
            preds = logits.argmax(dim=1)
            ys.extend(labels.cpu().numpy())
            ps.extend(preds.cpu().numpy())
    return total_loss / len(loader.dataset), accuracy_score(ys, ps), np.array(ys), np.array(ps)

def train_one_epoch(model, loader, optimizer):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        images = normalize_batch(images)

        # TODO 4：完成五个训练动作
        # 1. optimizer 清空梯度
        # 2. model 前向传播得到 logits
        # 3. criterion 计算 loss
        # 4. loss 反向传播
        # 5. optimizer 更新参数
        pass

        running_loss += loss.item() * len(labels)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += len(labels)
    return running_loss / total, correct / total

In [ ]:
EPOCHS = 2
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_state = None
best_val = -1

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer)
    val_loss, val_acc, _, _ = evaluate(model, val_loader)
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    if val_acc > best_val:
        best_val = val_acc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    print(f'Epoch {epoch+1}/{EPOCHS} | train loss {train_loss:.4f} | train acc {train_acc:.4f} | val acc {val_acc:.4f}')

model.load_state_dict(best_state)
model.to(device)

fig, ax1 = plt.subplots(figsize=(8, 4.5))
ax1.plot(range(1, EPOCHS+1), history['train_loss'], marker='o', label='train loss')
ax1.plot(range(1, EPOCHS+1), history['val_loss'], marker='o', label='validation loss')
ax1.set_xlabel('epoch')
ax1.set_ylabel('loss')
ax2 = ax1.twinx()
ax2.plot(range(1, EPOCHS+1), history['val_acc'], marker='s', linestyle='--', label='validation accuracy')
ax2.set_ylabel('accuracy')
lines = ax1.get_lines() + ax2.get_lines()
ax1.legend(lines, [l.get_label() for l in lines], loc='center right')
plt.title('Training history')
plt.tight_layout()
plt.savefig(WORKDIR/'task0_training_curve.png', dpi=180, bbox_inches='tight')
plt.show()

## 任务 5：测试集评价

载入验证集表现最好的模型后，在测试集上计算准确率、macro-F1、每类 F1 和混淆矩阵。混淆矩阵的行表示真实标签，列表示预测标签。


In [ ]:
# TODO 5：调用 evaluate，并计算指标
# test_loss, test_acc, y_true, y_pred = ...
# macro_f1 = ...
# per_class_f1 = ...
# cm = ...

test_loss = None
test_acc = None
y_true = None
y_pred = None
macro_f1 = None
per_class_f1 = None
cm = None

print('test accuracy =', test_acc)
print('macro F1 =', macro_f1)
print('per-class F1 =', per_class_f1)

assert cm.shape == (10, 10)
assert len(per_class_f1) == 10

In [ ]:
plt.figure(figsize=(7, 6))
plt.imshow(cm, cmap='viridis')
plt.colorbar(label='sample count')
plt.xticks(range(10))
plt.yticks(range(10))
plt.xlabel('predicted label')
plt.ylabel('true label')
plt.title('MNIST test confusion matrix')
for i in range(10):
    for j in range(10):
        if cm[i, j] > 0:
            plt.text(j, i, int(cm[i, j]), ha='center', va='center', fontsize=7,
                     color='white' if cm[i, j] > cm.max()*0.45 else 'black')
plt.tight_layout()
plt.savefig(WORKDIR/'task0_confusion_matrix.png', dpi=180, bbox_inches='tight')
plt.show()

report = classification_report(y_true, y_pred, digits=4)
print(report)

## 任务 6：噪声稳健性

向原始测试图像加入标准差为 0.25 的高斯噪声，并把结果裁剪回 `[0, 1]`。保持模型参数不变，再次计算准确率。写出性能变化及其含义。


In [ ]:
# TODO 6：使用 evaluate 的 noise_sigma 参数完成稳健性评价
noise_sigma = 0.25
noisy_loss = None
noisy_acc = None
accuracy_drop = None

print('clean accuracy =', test_acc)
print('noisy accuracy =', noisy_acc)
print('accuracy drop =', accuracy_drop)
assert noisy_acc <= 1 and noisy_acc >= 0

## 任务 7：单变量对照

只修改学习率，再运行一次短训练。将基线的 `1e-3` 改为 `1e-2`，其他代码、数据划分和训练轮数保持不变。

记录原设置和修改设置的最佳验证准确率。结论需要说明比较条件、指标变化和可能原因。一次只修改一个条件，才能把变化与该条件联系起来。


In [ ]:
# TODO 7：填写你的对照实验记录
comparison = {
    'changed_variable': None,
    'baseline_value': None,
    'new_value': None,
    'baseline_best_val_accuracy': float(best_val),
    'new_best_val_accuracy': None,
    'observation': None,
}
comparison

## 8. 保存结果

确保所有指标来自当前实际运行。若当前显示 `sklearn digits fallback`，结果用于检查 Notebook；完成实践时请在 Kaggle 中重新运行 MNIST，并在 `task0_result.json` 中核对 `dataset_name`。


In [ ]:
result = {
    'dataset_name': dataset_name,
    'random_seed': SEED,
    'train_samples': len(train_dataset),
    'validation_samples': len(val_dataset),
    'test_samples': len(test_dataset),
    'image_shape': list(image_shape),
    'train_mean': float(train_mean),
    'train_std': float(train_std),
    'model_name': 'SmallCNN',
    'parameter_count': int(sum(p.numel() for p in model.parameters())),
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'optimizer': 'Adam',
    'learning_rate': 1e-3,
    'best_validation_accuracy': float(best_val),
    'test_loss': float(test_loss),
    'test_accuracy': float(test_acc),
    'test_macro_f1': float(macro_f1),
    'per_class_f1': [float(x) for x in per_class_f1],
    'noise_sigma': noise_sigma,
    'noisy_test_accuracy': float(noisy_acc),
    'accuracy_drop': float(accuracy_drop),
    'comparison': comparison,
    'output_files': [
        'task0_data_visualization.png',
        'task0_training_curve.png',
        'task0_confusion_matrix.png',
        'task0_result.json'
    ]
}
with open(WORKDIR/'task0_result.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=2)
print(json.dumps(result, ensure_ascii=False, indent=2))

## 结果检查

完成后按顺序检查四个输出文件。数据图用于确认输入和标签读取正确；训练曲线用于判断训练是否发生；混淆矩阵用于定位类别错误；JSON 用于保存能够复现实验的设置与指标。可以在 Notebook 中用几句话记录数据、模型、测试表现、主要混淆、噪声影响和单变量对照结果。
